In [1]:
import sys
sys.path.append("../graph-matching-ppr-framework/")

In [2]:
from enum import Enum
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple
import networkx as nx

from models import PageRankSettings, DistanceMetricsSettings
from enums import SortMethod

In [3]:
class GraphModel(Enum):
    gnp = 0
    ba = 1

In [4]:
@dataclass(frozen=True)
class GraphFilenames:
    original: str = "original.edgelist"
    graph_a: str = "a.edgelist"
    graph_b: str = "b.edgelist"

In [5]:
@dataclass
class GraphParameters:
    graph_model: GraphModel
    n_nodes: int
    edge_removal_prob: float

In [6]:
@dataclass
class ProgressFileParameters:
    sort_method: SortMethod
    return_alpha: float
    amortization_sigma: float
    main_distance_importance_gamma: float

In [7]:
@dataclass
class ComparisonResult:
    name: str
    result_per_instance: list[float]

In [8]:
def build_graph_path(
    base_dir: Path,
    graph_params: GraphParameters,
    instance_idx: int
) -> Path:
    return Path(base_dir) / str(graph_params.graph_model.name) / str(graph_params.n_nodes) / str(graph_params.edge_removal_prob) / str(instance_idx)

In [9]:
def get_progress_output_dir(
    graph_params: GraphParameters,
    instance_idx: int
) -> Path:
    return build_graph_path(
        base_dir="../progress_output",
        graph_params=graph_params,
        instance_idx=instance_idx
    )

In [10]:
def get_progress_output_filename(
    graph_params: GraphParameters,
    instance_idx: int,
    progress_file_params: ProgressFileParameters
) -> Path:
    progress_output_dir = get_progress_output_dir(
        graph_params=graph_params,
        instance_idx=instance_idx
    )
    sort_method = progress_file_params.sort_method.name
    alpha = progress_file_params.return_alpha
    sigma = progress_file_params.amortization_sigma
    gamma = progress_file_params.main_distance_importance_gamma
    return progress_output_dir / f"progress-{sort_method}-alpha-{alpha}-sigma-{sigma}-gamma-{gamma}"

In [11]:
def get_graph_input_dir(
    graph_params: GraphParameters,
    instance_idx: int
) -> Path:
    return build_graph_path(
        base_dir="../input_graphs",
        graph_params=graph_params,
        instance_idx=instance_idx
    )

In [12]:
def get_input_graphs(
    graph_params: GraphParameters,
    instance_idx: int
) -> Tuple[nx.Graph, nx.Graph]:
    graph_path = get_graph_input_dir(
        graph_params=graph_params,
        instance_idx=instance_idx
    )
    graph_a = nx.read_edgelist(graph_path / GraphFilenames.graph_a)
    graph_b = nx.read_edgelist(graph_path / GraphFilenames.graph_b)
    return graph_a, graph_b

In [13]:
def get_original_graph(
    graph_params: GraphParameters,
    instance_idx: int
) -> nx.Graph:
    graph_path = get_graph_input_dir(
        graph_params=graph_params,
        instance_idx=instance_idx
    )
    original_graph = nx.read_edgelist(graph_path / GraphFilenames.original)
    return original_graph